In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# 1. 데이터 로드
file_path = "AUS_v2_Ready_To_Train.csv"
df = pd.read_csv(file_path, dtype={'COMPANY_ID_NORM': str})

# 2. 모델 재가동 (v3.0 진검승부 세팅)
leakage_cols = ['MORATORIUM_COUNT', 'MORATORIUM_OVERDUE_AMOUNT', 'ACCOUNT_SUSPENSION_COUNT', 'CARD_ACCOUNT_COUNT']
exclude_cols = ['COMPANY_ID', 'COMPANY_ID_NORM', 'TARGET_Y', 'TMP_Y'] + leakage_cols
features = [c for c in df.columns if c in df.select_dtypes(include=[np.number]).columns and c not in exclude_cols]

X = df[features]
y = df['TARGET_Y']

rf_v3 = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42)
rf_v3.fit(X, y)

# 3. AI 점수 산출 (100점 만점 기준)
probs = rf_v3.predict_proba(X)[:, 1]
df['AI_PROBABILITY'] = probs
df['AI_SCORE'] = (1 - probs) * 100

# 4. [진단 핵심] 사각지대 39개사 추출 및 리스크 원인 분석
# AI 점수가 30점 미만인 고위험군을 타겟팅합니다.
blind_spot_list = df[df['AI_SCORE'] < 30].sort_values(by='AI_SCORE').copy()

# 정상 기업들의 중앙값 (비교 기준)
normal_median = df[df['TARGET_Y'] == 0].median(numeric_only=True)

def analyze_risk(row):
    reasons = []
    # 1순위: 현금 고갈 (CASH_RATIO)
    if row['CASH_RATIO'] < normal_median['CASH_RATIO'] * 0.5:
        reasons.append(f"현금부족(정상의 {row['CASH_RATIO']/normal_median['CASH_RATIO']:.1%})")
    # 2순위: 유동성 스트레스
    if row['FE_LIQUIDITY_STRESS'] > normal_median['FE_LIQUIDITY_STRESS'] * 1.5:
        reasons.append("유동성 압박 심각")
    # 3순위: 공급망 고립
    if row['LINKED_PARTNERS'] <= 1:
        reasons.append("공급망 고립(거래처 1개 이하)")
    return " | ".join(reasons) if reasons else "복합 재무 리스크"

blind_spot_list['RISK_REASON'] = blind_spot_list.apply(analyze_risk, axis=1)

# 5. 최종 리스트 정리 및 출력
report_cols = ['COMPANY_ID_NORM', 'AI_SCORE', 'SALES_REVENUE', 'CASH_RATIO', 'LINKED_PARTNERS', 'RISK_REASON']
print(f"🚩 [Internal Deep-Scan] 사각지대 {len(blind_spot_list)}개사 진단 리포트 (상위 10개)")
print("-" * 110)
print(blind_spot_list[report_cols].head(10).to_string(index=False))

# 파일 저장
blind_spot_list[report_cols].to_csv("Final_BlindSpot_DeepScan.csv", index=False, encoding='utf-8-sig')

🚩 [Internal Deep-Scan] 사각지대 9개사 진단 리포트 (상위 10개)
--------------------------------------------------------------------------------------------------------------
COMPANY_ID_NORM  AI_SCORE  SALES_REVENUE  CASH_RATIO  LINKED_PARTNERS                                     RISK_REASON
           1195 15.642695      3966904.0    0.235776              0.0  현금부족(정상의 1.6%) | 유동성 압박 심각 | 공급망 고립(거래처 1개 이하)
            255 23.294862      5562982.0    2.174207              6.0                     현금부족(정상의 14.9%) | 유동성 압박 심각
           1140 23.790667       439724.0    1.081596              0.0  현금부족(정상의 7.4%) | 유동성 압박 심각 | 공급망 고립(거래처 1개 이하)
             80 27.406770      1373330.0    0.666735              2.0                      현금부족(정상의 4.6%) | 유동성 압박 심각
           1111 28.096687      2160828.0   24.775960              0.0                   유동성 압박 심각 | 공급망 고립(거래처 1개 이하)
           1251 28.186771       694670.0    4.723955              0.0 현금부족(정상의 32.3%) | 유동성 압박 심각 | 공급망 고립(거래처 1개 이하)
            273